In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')

# Load data
train_df = pd.read_csv('../data/processed/train_data_clean.csv')
test_df = pd.read_csv('../data/processed/test_data_clean.csv')

drop_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime',
             'readmitted_30d', 'readmitted_60d', 'readmitted_90d']

X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df['readmitted_30d']
X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df['readmitted_30d']

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

# Hyperparameter search space
param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 1.5, 2]
}

xgb_base = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc',
    verbosity=0,
    tree_method='hist'  # faster on large datasets
)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Starting hyperparameter search...")
search.fit(X_train, y_train)

print(f"\nBest params: {search.best_params_}")
print(f"Best CV AUROC: {search.best_score_:.4f}")

# Evaluate best model on test set
best_model = search.best_estimator_
y_test_pred = best_model.predict_proba(X_test)[:, 1]
test_auroc = roc_auc_score(y_test, y_test_pred)
print(f"Test AUROC: {test_auroc:.4f}")

if test_auroc >= 0.75:
    print("TARGET MET!")
else:
    print(f"Gap remaining: {0.75 - test_auroc:.4f}")

# Save best model
import joblib
joblib.dump(best_model, '../models/xgboost_tuned.pkl')
print("Model saved")

Starting hyperparameter search...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Best params: {'subsample': 0.8, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best CV AUROC: 0.7237
Test AUROC: 0.7197
Gap remaining: 0.0303
Model saved


In [2]:
train_df = pd.read_csv('../data/processed/train_data_clean.csv')
patient_admission_counts = train_df.groupby('subject_id')['hadm_id'].count()
print(f"Total patients: {train_df['subject_id'].nunique()}")
print(f"Total admissions: {len(train_df)}")
print(f"Avg admissions per patient: {patient_admission_counts.mean():.2f}")
print(f"Max admissions per patient: {patient_admission_counts.max()}")
print(f"Patients with only 1 admission: {(patient_admission_counts == 1).sum()}")
print(f"Patients with 2+ admissions: {(patient_admission_counts >= 2).sum()}")

Total patients: 150508
Total admissions: 324824
Avg admissions per patient: 2.16
Max admissions per patient: 95
Patients with only 1 admission: 88238
Patients with 2+ admissions: 62270


In [3]:
print(train_df.sort_values(['subject_id', 'admittime']).groupby('subject_id').head(3)[
    ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'readmitted_30d']
].head(20).to_string())

        subject_id   hadm_id            admittime            dischtime  readmitted_30d
233015    10000084  23052089  2160-11-21 01:56:00  2160-11-25 14:52:00               0
180383    10000690  26504700  2150-07-03 02:46:00  2150-07-07 15:42:00               0
181424    10000690  23280645  2150-09-16 19:48:00  2150-09-24 13:50:00               0
182070    10000690  25860671  2150-11-02 18:02:00  2150-11-12 13:45:00               0
127694    10000719  24558333  2140-04-15 00:14:00  2140-04-18 12:29:00               0
89496     10000764  27897940  2132-10-14 23:31:00  2132-10-19 16:30:00               0
162105    10000826  20032235  2146-12-05 19:07:00  2146-12-12 16:30:00               1
162289    10000826  21086876  2146-12-18 17:39:00  2146-12-24 19:55:00               1
162466    10000826  28289260  2146-12-31 00:43:00  2147-01-02 17:45:00               0
49107     10000883  25221576  2124-05-14 21:11:00  2124-05-22 10:40:00               0
36347     10000947  27880650  2121-05-09 10